# Forecasting, Riesgo y Asset Allocation — Modelo de Seis Factores de Fama-French
**Tesis de Magíster — Universidad Diego Portales**  
**Autores:** Álvaro Gatica, Fernanda Rojas, Luis Pizarro, Cristian Illanes  
**Notebook 01:** Descarga y Exploración de Datos

---

**Objetivo:** Descargar los factores Fama-French 6 (FF6) y los precios históricos de un universo de activos. Explorar estadísticas descriptivas, correlaciones y distribuciones de retornos.

## 0. Instalación de dependencias

In [ ]:
# Ejecutar solo la primera vez
# !pip install pandas numpy pandas_datareader yfinance statsmodels arch cvxpy scipy matplotlib seaborn jupyter

## 1. Imports y configuración global

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings

import pandas_datareader.data as web
import yfinance as yf

from datetime import datetime
from scipy import stats as sp_stats

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# ─── Parámetros globales ─────────────────────────────────────────
START = '2000-01-01'
END   = datetime.today().strftime('%Y-%m-%d')

print(f'Período de análisis: {START} → {END}')

## 2. Descarga de Factores Fama-French 6 (FF6)

Fuente: Kenneth French Data Library vía `pandas_datareader`.

In [ ]:
# Factores FF5 (Mkt-RF, SMB, HML, RMW, CMA) + RF
ff5_monthly = web.DataReader(
    'F-F_Research_Data_5_Factors_2x3',
    'famafrench',
    start=START
)[0] / 100

# Factor Momentum (WML)
mom_monthly = web.DataReader(
    'F-F_Momentum_Factor',
    'famafrench',
    start=START
)[0] / 100
mom_monthly.columns = ['WML']

# Unir en un solo DataFrame FF6
ff6 = ff5_monthly.join(mom_monthly, how='inner')
ff6.index = pd.to_datetime(ff6.index.to_timestamp())

print(f'Shape: {ff6.shape}')
print(f'Período: {ff6.index[0].date()} → {ff6.index[-1].date()}')
ff6.tail()

In [ ]:
# Estadísticas descriptivas de los factores (anualizadas)
factores = [c for c in ff6.columns if c != 'RF']
ff6_f = ff6[factores]

stats_ff6 = pd.DataFrame({
    'Media anual':       ff6_f.mean() * 12,
    'Volatilidad anual': ff6_f.std() * np.sqrt(12),
    'Sharpe':            (ff6_f.mean() * 12) / (ff6_f.std() * np.sqrt(12)),
    'Skewness':          ff6_f.skew(),
    'Kurtosis':          ff6_f.kurt()
})
stats_ff6

In [ ]:
# Retornos acumulados de cada factor
cumret = (1 + ff6_f).cumprod()

fig, ax = plt.subplots(figsize=(14, 6))
for col in cumret.columns:
    ax.plot(cumret.index, cumret[col], label=col, linewidth=1.5)

ax.set_title('Retornos Acumulados — Factores Fama-French 6', fontsize=14)
ax.set_ylabel('Valor (base 1)')
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

## 3. Universo de Activos — Descarga de Precios

Se utilizan acciones del S&P 500 como universo inicial. Modificar `TICKERS` según el universo final de la tesis.

In [ ]:
# ─── Universo de activos (modificar según tesis) ─────────────────
TICKERS = [
    'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META',   # Tech
    'JPM',  'BAC',  'GS',                        # Financials
    'JNJ',  'PFE',  'UNH',                       # Healthcare
    'XOM',  'CVX',                               # Energy
    'WMT',  'PG',   'KO',                        # Consumer Staples
    'SPY'                                         # Benchmark S&P500
]

# Descarga de precios ajustados
raw = yf.download(TICKERS, start=START, end=END, auto_adjust=True, progress=False)['Close']
raw.index = pd.to_datetime(raw.index)

print(f'Shape: {raw.shape}')
print(f'Período: {raw.index[0].date()} → {raw.index[-1].date()}')
raw.tail()

In [ ]:
# Retornos diarios y mensuales
ret_daily   = raw.pct_change().dropna()
ret_monthly = raw.resample('ME').last().pct_change().dropna()

print(f'Retornos diarios   — Shape: {ret_daily.shape}')
print(f'Retornos mensuales — Shape: {ret_monthly.shape}')

## 4. Estadísticas Descriptivas de los Activos

In [ ]:
# Alinear retornos con tasa libre de riesgo
common_idx = ret_monthly.index.intersection(ff6.index)
ret_m  = ret_monthly.loc[common_idx]
ff6_m  = ff6.loc[common_idx]
Rf_m   = ff6_m['RF']

# Estadísticas anualizadas
n = 12
stats_activos = pd.DataFrame({
    'Retorno anual':     ret_m.mean() * n,
    'Volatilidad anual': ret_m.std() * np.sqrt(n),
    'Sharpe':            (ret_m.mean() - Rf_m.mean()) * n / (ret_m.std() * np.sqrt(n)),
    'Skewness':          ret_m.skew(),
    'Kurtosis':          ret_m.kurt(),
    'Max Drawdown':      (raw / raw.cummax() - 1).min()
})
stats_activos.sort_values('Sharpe', ascending=False)

In [ ]:
# Mapa de calor de correlaciones (retornos mensuales)
corr = ret_m.corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax
)
ax.set_title('Correlación de Retornos Mensuales', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Distribución de retornos vs. normal teórica
ncols = 4
nrows = int(np.ceil(len(ret_m.columns) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 3))
axes = axes.flatten()

for i, ticker in enumerate(ret_m.columns):
    r = ret_m[ticker].dropna()
    axes[i].hist(r, bins=30, density=True, alpha=0.6, color='steelblue')
    xmin, xmax = axes[i].get_xlim()
    x = np.linspace(xmin, xmax, 100)
    axes[i].plot(x, sp_stats.norm.pdf(x, r.mean(), r.std()), 'r--', linewidth=1.5)
    jb_stat, jb_p = sp_stats.jarque_bera(r)
    axes[i].set_title(f'{ticker}  (JB p={jb_p:.3f})', fontsize=9)
    axes[i].tick_params(labelsize=7)

# Ocultar subplots vacíos
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Distribución de Retornos Mensuales vs. Normal', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Exportación de Datos

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)

ret_m.to_csv('../data/processed/retornos_mensuales.csv')
ff6_m.to_csv('../data/processed/factores_ff6_mensuales.csv')
ret_daily.to_csv('../data/processed/retornos_diarios.csv')

print('Archivos exportados:')
print('  data/processed/retornos_mensuales.csv')
print('  data/processed/factores_ff6_mensuales.csv')
print('  data/processed/retornos_diarios.csv')

---

## Próximos pasos

- **Notebook 02:** Regresiones de series de tiempo — estimación de betas y alphas del modelo FF6  
- **Notebook 03:** Construcción de la matriz de covarianza factorial

*Universidad Diego Portales — Magíster en Finanzas*